# TaaS Studio — Hypergraph Explorer Browser E2E (existing Colab Studio)

**Prerequisite:** first run `notebooks/colab/taas_core_studio_colab.ipynb` from top to bottom in this same runtime, including Launch Studio and Open TaaS Studio. This notebook reuses that exact Uvicorn process, generated `api_token`, and `proxy_url`. It does not clone another repository, start another server, bind another port, or replace the token.

The test uses the real Studio page and API, performs the metadata-only demo discovery through its UI, validates Core projection and browser interactions, and captures desktop/mobile screenshots. No MCP server is contacted and no tool is executed.


In [ ]:
# Reuse the running Studio launched by taas_core_studio_colab.ipynb.
# Do not start another API process or create/replace its token.
import os, pathlib
from IPython.display import display, Image

WORKDIR = pathlib.Path("/content/taas-provenance-discovery")
assert WORKDIR.is_dir(), "Run taas_core_studio_colab.ipynb first in this same Colab runtime."
assert (WORKDIR / "src" / "taas" / "studio" / "api.py").is_file()
assert "studio_process" in globals(), "The TaaS Core Studio launch cell must run first."
assert studio_process.poll() is None, "Studio is not running. Re-run its Launch Studio cell."
assert "api_token" in globals() and api_token, "The Core Studio notebook token is unavailable."
assert "proxy_url" in globals() and proxy_url, "Run the Open TaaS Studio cell first."

TAAS_STUDIO_URL = proxy_url
os.environ["TAAS_STUDIO_URL"] = TAAS_STUDIO_URL
os.environ["TAAS_STUDIO_API_TOKEN"] = api_token
os.environ["TAAS_E2E_ARTIFACT_DIR"] = "/content/taas-e2e-artifacts"
print("Reusing Studio process PID:", studio_process.pid)
print("Existing proxy URL:", TAAS_STUDIO_URL)
print("Existing API token: present (value hidden)")
%cd /content/taas-provenance-discovery



In [ ]:
!pip -q install playwright
!playwright install --with-deps chromium
!python -m compileall -q src/taas tests/test_studio_browser_e2e.py


In [ ]:
!pytest -q tests/test_retrieval_degenerate.py tests/test_studio_hypergraph_explorer.py tests/test_studio_navigation_shell.py tests/test_studio_professional_ux.py tests/test_hypergraph_projection.py tests/test_engine_certificate_gate.py tests/test_studio_browser_e2e.py -s


In [ ]:
from pathlib import Path
artifacts = sorted(Path(os.environ['TAAS_E2E_ARTIFACT_DIR']).glob('*.png'))
assert len(artifacts) >= 2, 'Expected desktop and mobile screenshots.'
for artifact in artifacts:
    print(artifact.name)
    display(Image(filename=str(artifact)))


## Runtime reuse and validation boundary

- Reuses server process, token and proxy URL created by `taas_core_studio_colab.ipynb`.
- Does not reinstall TaaS, create a token, launch Uvicorn or call `proxyPort`.
- Chromium visits the existing Studio proxy URL and authenticates with the existing notebook token.
- Discovery must return `taas.hypergraph-projection/v1` and `execution_performed=false`.
- Exercises search, SVG keyboard selection, accessible entity list and inspector.
- Runs desktop and mobile viewport checks and produces screenshot artifacts.
- Asserts no `/api/v1/mcp/*` browser requests.

A pass validates this browser journey against the current Colab process only. It does not prove production identity, live MCP execution, authorization, cross-browser compatibility, screen-reader certification or deployment resilience. Keep Colab output distinct from GitHub Actions status.
